# A-001 Complete Enterprise RAG Project
## LlamaIndex + Databricks + Persistent Index + Persistent Memory + Governance + Gradio

This notebook puts together the pieces learned so far into one end-to-end project.

### What we are building

```text
A-001 PDF documents
        ↓
LlamaIndex loader
        ↓
Chunking
        ↓
Databricks embeddings
        ↓
Persistent vector index
        ↓
Retrieve candidate chunks
        ↓
Governance filter
        ↓
Approved / relevant context only
        ↓
Persistent conversation memory
        +
System prompt
        +
Retrieved evidence
        ↓
Low-temperature LLM
        ↓
Cited answer
        ↓
Simple Gradio UI
```

### Core governance example

If the user asks:

> What is the approved inspection procedure for A-001?

the system should use **A001-PROC-INS-001-V2** as the current approved inspection procedure.

It should **not** use:

- **A001-PROC-INS-001-V1** as current guidance because it is SUPERSEDED.
- **A001-PROC-INS-001-V3D** as current guidance because it is DRAFT.

The user can still ask explicitly about historical or draft versions.


## 0. Before you run

This notebook assumes the 9 A-001 PDFs are available under:

```text
/Workspace/Nuclear_Enterprise_360/A001 Documents
```

The persistent LlamaIndex store and persistent chat memory will be saved under:

```text
/Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store
```

If your paths are different, change them in the configuration cell below.

> **Training note:** all A-001 documents in this project are synthetic training material, not operational engineering instructions.


In [0]:
%pip install -q -U \
    gradio \
    llama-index \
    llama-index-readers-file \
    llama-index-llms-databricks \
    llama-index-embeddings-databricks \
    pypdf

dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Imports

We keep the application simple:

- `SimpleDirectoryReader` loads PDFs.
- `SentenceSplitter` chunks the documents.
- `DatabricksEmbedding` creates vectors.
- `VectorStoreIndex` stores and retrieves document chunks.
- `StorageContext` lets us persist and reload the index.
- Gradio provides the question-and-answer UI.


In [0]:
import json
import os
import re
from pathlib import Path
from typing import Dict, List, Tuple

import gradio as gr

from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
    StorageContext,
    VectorStoreIndex,
    load_index_from_storage,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import NodeWithScore
from llama_index.embeddings.databricks import DatabricksEmbedding
from llama_index.llms.databricks import Databricks

print("Imports loaded successfully.")


Imports loaded successfully.


## 2. Configuration

Think of this as the control panel for the whole project.

### Models
- **Chat model** generates the answer.
- **Embedding model** converts text into vectors for semantic search.

### RAG parameters
- `CHUNK_SIZE = 400`
- `CHUNK_OVERLAP = 100`
- Retrieve up to 10 candidates first.
- After governance filtering, keep up to 5 context chunks.

### Temperature
A low temperature is used because this is an enterprise evidence-grounded use case.


In [0]:
# ============================================================
# CONFIGURATION
# ============================================================

DOCUMENT_FOLDER = "/Workspace/Nuclear_Enterprise_360/A001 Documents"

# Persist the index in a Unity Catalog Volume so it survives restarts.
PERSIST_DIR = Path(
    "/Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store"
)

MEMORY_DIR = PERSIST_DIR / "memory"

# Known working endpoints from the training environment.
# If your workspace exposes a stronger chat endpoint, change CHAT_MODEL only.
#system.ai.gpt-oss-120b 
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-qwen3-embedding-0-6b"

TEMPERATURE = 0.1

CHUNK_SIZE = 600
CHUNK_OVERLAP = 100

RETRIEVAL_TOP_K = 10
FINAL_CONTEXT_K = 5

# Keep the latest few turns as persistent conversation context.
MEMORY_TURNS = 6

print("Document folder :", DOCUMENT_FOLDER)
print("Persistent store:", PERSIST_DIR)
print("Chat model      :", CHAT_MODEL)
print("Embedding model :", EMBED_MODEL)
print("Chunk size      :", CHUNK_SIZE)
print("Chunk overlap   :", CHUNK_OVERLAP)
print("Retrieve top-k  :", RETRIEVAL_TOP_K)
print("Final context-k :", FINAL_CONTEXT_K)


Document folder : /Workspace/Nuclear_Enterprise_360/A001 Documents
Persistent store: /Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store
Chat model      : databricks-meta-llama-3-3-70b-instruct
Embedding model : databricks-qwen3-embedding-0-6b
Chunk size      : 600
Chunk overlap   : 100
Retrieve top-k  : 10
Final context-k : 5


## 3. System prompt

The prompt does four important things:

1. **Grounding** — use only retrieved A-001 evidence.
2. **Authority control** — current approved inspection questions must use V2.
3. **Evidence discipline** — observations, interpretation, and human review stay separate.
4. **Citations** — material claims should include the relevant document ID.

The system prompt is intentionally strict because retrieval similarity alone does not establish document authority.


In [0]:
SYSTEM_PROMPT = """
You are the A-001 Asset Reliability RAG Assistant for a synthetic training environment.

MISSION
Answer the user's question using only the retrieved A-001 training evidence supplied to you.
Be concise, evidence-based, and explicit about uncertainty.
Never invent measurements, approvals, work completion, diagnoses, repair instructions,
shutdown decisions, restart instructions, or operational authorizations.

SOURCE AUTHORITY RULES
1. For CURRENT or APPROVED inspection guidance, A001-PROC-INS-001-V2 is the authoritative procedure.
2. A001-PROC-INS-001-V1 is SUPERSEDED. Do not use it as current guidance.
3. A001-PROC-INS-001-V3D is DRAFT. Do not use it as current guidance.
4. You may discuss V1 or V3D only when the user explicitly asks about historical,
   superseded, draft, proposed, or version-comparison content.
5. A higher or newer-looking version number does not make a document authoritative.
6. For enterprise escalation and governance, prefer REL-POL-001.
7. For formal inspection workflow, prefer A001-PROC-INS-001-V2 over the operating guide,
   troubleshooting guide, field report, or condition monitoring report.
8. For questions about what happened during August, use A001-CMR-2026-08 and
   other relevant retrieved evidence.
9. For what work is currently requested, use WO-2026-0817.
10. For field observations, use CR-2026-0819-A001, but do not convert the observation
    into a root-cause diagnosis.

EVIDENCE DISCIPLINE
- Separate OBSERVED FACTS from INTERPRETATION and NEXT HUMAN REVIEW.
- A single high reading is not, by itself, a diagnosis.
- If evidence conflicts, preserve the conflict and state it clearly.
- If evidence is missing, stale, or insufficient, say so.
- Do not infer work completion merely because a work order exists.
- Do not convert an AI risk score, similarity score, or generated summary into maintenance authorization.
- Memory is conversation context, not evidence.
- If conversation memory conflicts with retrieved documents, retrieved documents win.
- Do not give shutdown, restart, component replacement, or other operational instructions.

CITATION RULES
- Cite every material factual claim with the document ID in square brackets.
  Example: [A001-PROC-INS-001-V2]
- When multiple documents support a claim, cite each relevant document.
- Never cite a document that is not present in the retrieved context.
- For a question about the approved/current inspection procedure, the answer must cite:
  [A001-PROC-INS-001-V2]

ANSWER STYLE
Use this structure when useful:

Answer:
<direct answer>

Evidence:
- <fact> [DOCUMENT-ID]
- <fact> [DOCUMENT-ID]

Interpretation:
<what the evidence supports without overstating>

Next human review:
<only if the evidence indicates review/escalation>

If the retrieved evidence is insufficient, say:
"I do not have enough approved evidence in the retrieved context to answer this safely."
""".strip()

print(SYSTEM_PROMPT[:1500])


You are the A-001 Asset Reliability RAG Assistant for a synthetic training environment.

MISSION
Answer the user's question using only the retrieved A-001 training evidence supplied to you.
Be concise, evidence-based, and explicit about uncertainty.
Never invent measurements, approvals, work completion, diagnoses, repair instructions,
shutdown decisions, restart instructions, or operational authorizations.

SOURCE AUTHORITY RULES
1. For CURRENT or APPROVED inspection guidance, A001-PROC-INS-001-V2 is the authoritative procedure.
2. A001-PROC-INS-001-V1 is SUPERSEDED. Do not use it as current guidance.
3. A001-PROC-INS-001-V3D is DRAFT. Do not use it as current guidance.
4. You may discuss V1 or V3D only when the user explicitly asks about historical,
   superseded, draft, proposed, or version-comparison content.
5. A higher or newer-looking version number does not make a document authoritative.
6. For enterprise escalation and governance, prefer REL-POL-001.
7. For formal inspection wo

## 4. Document governance registry

Vector similarity answers:

> “Which text looks similar to the question?”

Enterprise RAG also needs to answer:

> “Which source is actually allowed to act as current authority?”

So we attach metadata to every document:

- `document_id`
- `version`
- `approval_status`
- `document_role`

This gives us a governance layer **before** the LLM generates an answer.


In [0]:
DOCUMENT_REGISTRY = [
    {
        "pattern": "REL-POL-001",
        "document_id": "REL-POL-001",
        "version": "2.1",
        "approval_status": "APPROVED",
        "document_role": "enterprise reliability and escalation policy",
    },
    {
        "pattern": "A001-OPS-001",
        "document_id": "A001-OPS-001",
        "version": "1.4",
        "approval_status": "APPROVED",
        "document_role": "operating and usage guide",
    },
    {
        "pattern": "A001-PROC-INS-001-V1",
        "document_id": "A001-PROC-INS-001-V1",
        "version": "1.0",
        "approval_status": "SUPERSEDED",
        "document_role": "historical inspection procedure",
    },
    {
        "pattern": "A001-PROC-INS-001-V2",
        "document_id": "A001-PROC-INS-001-V2",
        "version": "2.0",
        "approval_status": "APPROVED",
        "document_role": "current approved inspection procedure",
    },
    {
        "pattern": "A001-PROC-INS-001-V3D",
        "document_id": "A001-PROC-INS-001-V3D",
        "version": "3.0-draft",
        "approval_status": "DRAFT",
        "document_role": "draft proposed inspection procedure",
    },
    {
        "pattern": "A001-TSG-001",
        "document_id": "A001-TSG-001",
        "version": "1.2",
        "approval_status": "APPROVED",
        "document_role": "troubleshooting and diagnostic guide",
    },
    {
        "pattern": "A001-CMR-2026-08",
        "document_id": "A001-CMR-2026-08",
        "version": "1.0",
        "approval_status": "APPROVED",
        "document_role": "August condition monitoring report",
    },
    {
        "pattern": "WO-2026-0817",
        "document_id": "WO-2026-0817",
        "version": "1.0",
        "approval_status": "OPEN",
        "document_role": "inspection work order",
    },
    {
        "pattern": "CR-2026-0819-A001",
        "document_id": "CR-2026-0819-A001",
        "version": "1.0",
        "approval_status": "APPROVED",
        "document_role": "field condition report",
    },
]

def metadata_for_filename(file_name: str) -> Dict[str, str]:
    """
    Map the actual PDF filename to governance metadata.

    We use 'contains' matching because the uploaded filenames may include
    suffixes such as (1) or (1)(1).
    """
    for item in DOCUMENT_REGISTRY:
        if item["pattern"].lower() in file_name.lower():
            return dict(item)

    # Safe fallback: unknown sources are not silently treated as approved.
    return {
        "document_id": Path(file_name).stem,
        "version": "unknown",
        "approval_status": "UNKNOWN",
        "document_role": "unclassified source",
    }

for item in DOCUMENT_REGISTRY:
    print(
        item["document_id"],
        "->",
        item["approval_status"],
        "|",
        item["document_role"],
    )


REL-POL-001 -> APPROVED | enterprise reliability and escalation policy
A001-OPS-001 -> APPROVED | operating and usage guide
A001-PROC-INS-001-V1 -> SUPERSEDED | historical inspection procedure
A001-PROC-INS-001-V2 -> APPROVED | current approved inspection procedure
A001-PROC-INS-001-V3D -> DRAFT | draft proposed inspection procedure
A001-TSG-001 -> APPROVED | troubleshooting and diagnostic guide
A001-CMR-2026-08 -> APPROVED | August condition monitoring report
WO-2026-0817 -> OPEN | inspection work order
CR-2026-0819-A001 -> APPROVED | field condition report


## 5. Configure Databricks LLM + embeddings

LlamaIndex `Settings` acts like a central configuration point.

```text
Settings.llm
    → model that writes the final answer

Settings.embed_model
    → model that converts text into embeddings

Settings.text_splitter
    → controls how the PDFs are chunked
```

`TOP_K` is not a global `Settings` property here.  
We apply it later at retrieval time.


In [0]:
# Get the current Databricks workspace URL and notebook API token.
API_ROOT = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiUrl()
    .get()
)

API_TOKEN = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

SERVING_ENDPOINT = f"{API_ROOT}/serving-endpoints"

# Low-temperature LLM for controlled, evidence-grounded answers.
llm = Databricks(
    model=CHAT_MODEL,
    api_key=API_TOKEN,
    api_base=SERVING_ENDPOINT,
    temperature=TEMPERATURE,
)

# Embedding model for semantic retrieval.
embed_model = DatabricksEmbedding(
    model=EMBED_MODEL,
    api_key=API_TOKEN,
    endpoint=SERVING_ENDPOINT,
)

Settings.llm = llm
Settings.embed_model = embed_model

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

Settings.text_splitter = splitter

print("Databricks models and LlamaIndex settings configured.")


Databricks models and LlamaIndex settings configured.


## 6. Validate the PDF folder

Before indexing, confirm the PDFs are visible.

This also gives students a nice sanity check before doing anything expensive.


In [0]:
document_path = Path(DOCUMENT_FOLDER)

if not document_path.is_dir():
    raise FileNotFoundError(
        f"Document folder not found: {DOCUMENT_FOLDER}"
    )

pdf_files = sorted(document_path.rglob("*.pdf"))

print(f"PDF files found: {len(pdf_files)}")
for i, path in enumerate(pdf_files, 1):
    print(f"{i:02d}. {path.name}")

if not pdf_files:
    raise RuntimeError("No PDFs found. Check DOCUMENT_FOLDER.")


PDF files found: 9
01. 01_REL-POL-001_Enterprise_Asset_Reliability_and_Escalation_Policy.pdf
02. 02_A001-OPS-001_Operating_and_Usage_Guide.pdf
03. 03_A001-PROC-INS-001-V1_Superseded_Inspection_Procedure.pdf
04. 04_A001-PROC-INS-001-V2_Approved_Inspection_and_PM_Procedure.pdf
05. 05_A001-PROC-INS-001-V3D_Draft_Procedure.pdf
06. 06_A001-TSG-001_Troubleshooting_and_Diagnostic_Guide.pdf
07. 07_A001-CMR-2026-08_Condition_Monitoring_Report.pdf
08. 08_WO-2026-0817_A001_Inspection_Work_Order.pdf
09. 09_CR-2026-0819_A001_Field_Condition_Report.pdf


## 7. Build or reload the persistent vector index

### First run

```text
PDFs
 ↓
load
 ↓
attach governance metadata
 ↓
chunk
 ↓
embed
 ↓
vector index
 ↓
persist to disk
```

### Later runs

```text
persistent directory
 ↓
load existing index
```

This means we do **not** need to recreate all embeddings every time the notebook restarts.


In [0]:
import time


def persistent_index_exists() -> bool:
    """Check whether the minimum LlamaIndex persistence files exist."""
    return (
        (PERSIST_DIR / "docstore.json").exists()
        and (PERSIST_DIR / "index_store.json").exists()
    )


def build_index() -> VectorStoreIndex:
    """
    Load all PDFs, attach governance metadata, build the vector index,
    and save it to the persistent directory.
    """
    documents = SimpleDirectoryReader(
        input_dir=DOCUMENT_FOLDER,
        recursive=True,
        required_exts=[".pdf"],
    ).load_data()

    if not documents:
        raise RuntimeError(f"No PDF documents found in {DOCUMENT_FOLDER}")

    # Each loaded page/document gets governance metadata.
    for doc in documents:
        file_name = doc.metadata.get("file_name", "")
        doc.metadata.update(metadata_for_filename(file_name))

    print("Loaded document objects/pages:", len(documents))

    # Split documents into nodes, then insert one at a time to respect
    # workspace QPS limits on the embedding endpoint.
    nodes = splitter.get_nodes_from_documents(documents)
    print("Total nodes to embed:", len(nodes))

    index = VectorStoreIndex(nodes=[])

    for i, node in enumerate(nodes):
        index.insert_nodes([node])
        if (i + 1) % 5 == 0:
            print(f"  embedded {i + 1}/{len(nodes)} nodes")
        time.sleep(3)

    PERSIST_DIR.mkdir(parents=True, exist_ok=True)
    index.storage_context.persist(
        persist_dir=str(PERSIST_DIR)
    )

    print("Persistent index created at:", PERSIST_DIR)
    return index


def load_existing_index() -> VectorStoreIndex:
    """Reload the previously persisted LlamaIndex store."""
    storage_context = StorageContext.from_defaults(
        persist_dir=str(PERSIST_DIR)
    )

    loaded_index = load_index_from_storage(storage_context)
    print("Persistent index loaded from:", PERSIST_DIR)
    return loaded_index


def load_or_build_index() -> VectorStoreIndex:
    if persistent_index_exists():
        return load_existing_index()

    return build_index()


index = load_or_build_index()

retriever = index.as_retriever(
    similarity_top_k=RETRIEVAL_TOP_K
)

print("Retriever ready.")


Loaded document objects/pages: 17
Total nodes to embed: 32


2026-09-22 10:15:37,609 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:15:41,090 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:15:44,521 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:15:47,816 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:15:51,364 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"


  embedded 5/32 nodes


2026-09-22 10:15:54,952 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:15:58,418 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:16:01,570 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:16:04,720 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:16:08,082 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"


  embedded 10/32 nodes


...

  embedded 25/32 nodes


2026-09-22 10:17:02,168 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:17:05,636 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:17:09,116 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:17:12,272 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:17:15,810 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"


  embedded 30/32 nodes


2026-09-22 10:17:19,120 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:17:22,433 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"


Persistent index created at: /Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store
Retriever ready.


## 8. Optional: inspect indexed metadata

This is useful for teaching and debugging.

You should see values such as:

```text
A001-PROC-INS-001-V1  → SUPERSEDED
A001-PROC-INS-001-V2  → APPROVED
A001-PROC-INS-001-V3D → DRAFT
```


In [0]:
seen = {}

for node in index.storage_context.docstore.docs.values():
    meta = getattr(node, "metadata", {}) or {}
    doc_id = meta.get("document_id")
    if doc_id:
        seen[doc_id] = {
            "status": meta.get("approval_status"),
            "version": meta.get("version"),
            "role": meta.get("document_role"),
        }

for doc_id, values in sorted(seen.items()):
    print(
        f"{doc_id:28s}",
        "|",
        f"{str(values['status']):10s}",
        "|",
        f"v{values['version']}",
        "|",
        values["role"],
    )


09_CR-2026-0819_A001_Field_Condition_Report | UNKNOWN    | vunknown | unclassified source
A001-CMR-2026-08             | APPROVED   | v1.0 | August condition monitoring report
A001-OPS-001                 | APPROVED   | v1.4 | operating and usage guide
A001-PROC-INS-001-V1         | SUPERSEDED | v1.0 | historical inspection procedure
A001-PROC-INS-001-V2         | APPROVED   | v2.0 | current approved inspection procedure
A001-PROC-INS-001-V3D        | DRAFT      | v3.0-draft | draft proposed inspection procedure
A001-TSG-001                 | APPROVED   | v1.2 | troubleshooting and diagnostic guide
REL-POL-001                  | APPROVED   | v2.1 | enterprise reliability and escalation policy
WO-2026-0817                 | OPEN       | v1.0 | inspection work order


## 9. Governed retrieval

This is the most important enterprise layer.

### Normal retrieval

We ask the vector index for the most similar chunks.

### Governance

Before those chunks reach the LLM:

- DRAFT documents are removed by default.
- SUPERSEDED documents are removed by default.
- If the user explicitly asks about draft/history, those sources are allowed.
- For a current/approved inspection-procedure question, V2 is explicitly forced into context.

This is a practical example of:

> **Relevance is not the same as authority.**


In [0]:
def asks_for_draft(question: str) -> bool:
    q = question.lower()
    return any(
        term in q
        for term in [
            "draft",
            "v3",
            "3.0-draft",
            "proposed",
            "future version",
        ]
    )


def asks_for_history(question: str) -> bool:
    q = question.lower()
    return any(
        term in q
        for term in [
            "superseded",
            "historical",
            "history",
            "v1",
            "version 1",
            "old procedure",
            "previous procedure",
        ]
    )


def asks_for_current_inspection_procedure(question: str) -> bool:
    q = question.lower()

    has_procedure = any(
        term in q
        for term in [
            "procedure",
            "inspection workflow",
            "inspection process",
        ]
    )

    current_signal = any(
        term in q
        for term in [
            "approved",
            "current",
            "use",
            "should follow",
            "applies",
            "applicable",
        ]
    )

    inspection_signal = any(
        term in q
        for term in [
            "inspection",
            "vibration",
            "abnormal condition",
        ]
    )

    return has_procedure and (current_signal or inspection_signal)


def source_key(item: NodeWithScore) -> str:
    meta = item.node.metadata or {}
    return (
        f"{meta.get('document_id', '')}|"
        f"{item.node.node_id}"
    )


def nodes_from_document(
    document_id: str,
    limit: int = 2,
) -> List[NodeWithScore]:
    """
    Fetch nodes from a specific authoritative source already stored
    inside the persistent docstore.
    """
    results = []

    for node in index.storage_context.docstore.docs.values():
        meta = getattr(node, "metadata", {}) or {}

        if meta.get("document_id") == document_id:
            results.append(
                NodeWithScore(
                    node=node,
                    score=1.0,
                )
            )

            if len(results) >= limit:
                break

    return results


def apply_governance(
    question: str,
    retrieved: List[NodeWithScore],
) -> List[NodeWithScore]:

    allow_draft = asks_for_draft(question)
    allow_history = asks_for_history(question)
    current_inspection = asks_for_current_inspection_procedure(question)

    filtered = []

    for item in retrieved:
        meta = item.node.metadata or {}
        status = str(
            meta.get("approval_status", "UNKNOWN")
        ).upper()

        # Reject draft guidance unless user explicitly asks for it.
        if status == "DRAFT" and not allow_draft:
            continue

        # Reject superseded guidance unless user asks for history/comparison.
        if status == "SUPERSEDED" and not allow_history:
            continue

        filtered.append(item)

    # Current inspection question:
    # force the authoritative V2 source into final context.
    if current_inspection:
        approved_v2 = nodes_from_document(
            "A001-PROC-INS-001-V2",
            limit=2,
        )

        combined = approved_v2 + filtered

        # Remove duplicate nodes.
        seen_keys = set()
        deduped = []

        for item in combined:
            key = source_key(item)

            if key not in seen_keys:
                seen_keys.add(key)
                deduped.append(item)

        filtered = deduped

    return filtered[:FINAL_CONTEXT_K]


print("Governance functions ready.")


Governance functions ready.


## 10. Test the governance layer before generation

This is a useful debugging habit.

We first test retrieval and governance **without** asking the LLM to generate anything.


In [0]:
def inspect_retrieval(question: str):
    raw_nodes = retriever.retrieve(question)
    governed_nodes = apply_governance(question, raw_nodes)

    print("QUESTION")
    print(question)

    print("\nRAW RETRIEVAL")
    print("-" * 80)

    for i, item in enumerate(raw_nodes, 1):
        meta = item.node.metadata or {}
        print(
            i,
            meta.get("document_id"),
            "|",
            meta.get("approval_status"),
            "| score:",
            round(item.score, 4) if item.score is not None else "n/a",
        )

    print("\nAFTER GOVERNANCE")
    print("-" * 80)

    for i, item in enumerate(governed_nodes, 1):
        meta = item.node.metadata or {}
        print(
            i,
            meta.get("document_id"),
            "|",
            meta.get("approval_status"),
            "| score:",
            round(item.score, 4) if item.score is not None else "n/a",
        )


inspect_retrieval(
    "What is the approved inspection procedure for reviewing rising vibration on A-001?"
)


2026-09-22 10:17:58,277 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"


QUESTION
What is the approved inspection procedure for reviewing rising vibration on A-001?

RAW RETRIEVAL
--------------------------------------------------------------------------------
1 A001-PROC-INS-001-V1 | SUPERSEDED | score: 0.7722
2 A001-CMR-2026-08 | APPROVED | score: 0.6782
3 WO-2026-0817 | OPEN | score: 0.6775
4 09_CR-2026-0819_A001_Field_Condition_Report | UNKNOWN | score: 0.6606
5 A001-TSG-001 | APPROVED | score: 0.6595
6 A001-PROC-INS-001-V2 | APPROVED | score: 0.6488
7 A001-PROC-INS-001-V2 | APPROVED | score: 0.6467
8 WO-2026-0817 | OPEN | score: 0.632
9 A001-CMR-2026-08 | APPROVED | score: 0.62
10 A001-PROC-INS-001-V1 | SUPERSEDED | score: 0.6099

AFTER GOVERNANCE
--------------------------------------------------------------------------------
1 A001-PROC-INS-001-V2 | APPROVED | score: 1.0
2 A001-PROC-INS-001-V2 | APPROVED | score: 1.0
3 A001-CMR-2026-08 | APPROVED | score: 0.6782
4 WO-2026-0817 | OPEN | score: 0.6775
5 09_CR-2026-0819_A001_Field_Condition_Report | UNK

### Expected behavior

For the approved/current inspection question:

- **A001-PROC-INS-001-V2** should appear in the governed context.
- **A001-PROC-INS-001-V1** should not be treated as current guidance.
- **A001-PROC-INS-001-V3D** should not be treated as current guidance.

Now compare that with a question explicitly asking about a draft or historical version.


In [0]:
inspect_retrieval(
    "What changes are proposed in the V3 draft inspection procedure?"
)

print("\n" + "=" * 100 + "\n")

inspect_retrieval(
    "How did the superseded V1 inspection procedure differ from the current procedure?"
)


2026-09-22 10:18:06,572 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"


QUESTION
What changes are proposed in the V3 draft inspection procedure?

RAW RETRIEVAL
--------------------------------------------------------------------------------
1 A001-PROC-INS-001-V3D | DRAFT | score: 0.6823
2 A001-PROC-INS-001-V1 | SUPERSEDED | score: 0.594
3 A001-PROC-INS-001-V2 | APPROVED | score: 0.5666
4 A001-PROC-INS-001-V3D | DRAFT | score: 0.5629
5 A001-PROC-INS-001-V3D | DRAFT | score: 0.5521
6 A001-PROC-INS-001-V1 | SUPERSEDED | score: 0.5483
7 A001-PROC-INS-001-V2 | APPROVED | score: 0.5471
8 WO-2026-0817 | OPEN | score: 0.52
9 A001-PROC-INS-001-V2 | APPROVED | score: 0.5181
10 WO-2026-0817 | OPEN | score: 0.5164

AFTER GOVERNANCE
--------------------------------------------------------------------------------
1 A001-PROC-INS-001-V2 | APPROVED | score: 1.0
2 A001-PROC-INS-001-V2 | APPROVED | score: 1.0
3 A001-PROC-INS-001-V3D | DRAFT | score: 0.6823
4 A001-PROC-INS-001-V2 | APPROVED | score: 0.5666
5 A001-PROC-INS-001-V3D | DRAFT | score: 0.5629




2026-09-22 10:18:06,888 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"


QUESTION
How did the superseded V1 inspection procedure differ from the current procedure?

RAW RETRIEVAL
--------------------------------------------------------------------------------
1 A001-PROC-INS-001-V1 | SUPERSEDED | score: 0.6017
2 A001-PROC-INS-001-V3D | DRAFT | score: 0.5988
3 A001-PROC-INS-001-V1 | SUPERSEDED | score: 0.5719
4 A001-PROC-INS-001-V2 | APPROVED | score: 0.5275
5 A001-PROC-INS-001-V2 | APPROVED | score: 0.4811
6 WO-2026-0817 | OPEN | score: 0.4801
7 A001-PROC-INS-001-V3D | DRAFT | score: 0.4762
8 09_CR-2026-0819_A001_Field_Condition_Report | UNKNOWN | score: 0.4758
9 A001-PROC-INS-001-V3D | DRAFT | score: 0.467
10 A001-PROC-INS-001-V2 | APPROVED | score: 0.461

AFTER GOVERNANCE
--------------------------------------------------------------------------------
1 A001-PROC-INS-001-V2 | APPROVED | score: 1.0
2 A001-PROC-INS-001-V2 | APPROVED | score: 1.0
3 A001-PROC-INS-001-V1 | SUPERSEDED | score: 0.6017
4 A001-PROC-INS-001-V1 | SUPERSEDED | score: 0.5719
5 A001-PR

## 11. Persistent conversation memory

We now add memory.

This project keeps recent conversation turns in a JSON file.

Example:

```text
a001_rag_store/
    docstore.json
    index_store.json
    ...
    memory/
        demo.json
        user_123.json
```

### Critical rule

> **Memory is context, not evidence.**

Memory can help interpret:

> “What about the previous one?”

But factual claims still need to come from retrieved documents.


In [0]:
def safe_memory_key(value: str) -> str:
    """Create a filesystem-safe conversation ID."""
    value = (value or "demo").strip()
    value = re.sub(r"[^A-Za-z0-9_-]+", "_", value)
    return value[:80] or "demo"


def memory_path(memory_key: str) -> Path:
    MEMORY_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    return MEMORY_DIR / f"{safe_memory_key(memory_key)}.json"


def load_memory(memory_key: str) -> List[Dict[str, str]]:
    path = memory_path(memory_key)

    if not path.exists():
        return []

    try:
        data = json.loads(
            path.read_text(encoding="utf-8")
        )

        history = data.get("history", [])

        if isinstance(history, list):
            return history[-(MEMORY_TURNS * 2):]

    except Exception as exc:
        print("Memory read warning:", exc)

    return []


def save_memory(
    memory_key: str,
    history: List[Dict[str, str]],
) -> None:
    path = memory_path(memory_key)

    payload = {
        "history": history[-(MEMORY_TURNS * 2):]
    }

    path.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def clear_memory(memory_key: str):
    path = memory_path(memory_key)

    if path.exists():
        path.unlink()

    return []


def memory_as_text(
    history: List[Dict[str, str]]
) -> str:

    if not history:
        return "(No prior conversation memory.)"

    lines = []

    for item in history[-(MEMORY_TURNS * 2):]:
        role = item.get("role", "user").upper()
        content = item.get("content", "")

        lines.append(
            f"{role}: {content}"
        )

    return "\n".join(lines)


print("Persistent memory functions ready.")


Persistent memory functions ready.


## 12. Build the governed evidence context

We do not send raw retrieved objects directly to the LLM.

Instead, we create a clear evidence package containing:

- source number
- document ID
- approval status
- version
- document role
- retrieved text

This helps the model understand both **content** and **authority**.


In [0]:
def build_context(
    nodes: List[NodeWithScore],
) -> Tuple[str, str]:

    context_blocks = []
    source_lines = []

    for i, item in enumerate(nodes, 1):
        meta = item.node.metadata or {}

        document_id = meta.get(
            "document_id",
            "UNKNOWN",
        )

        status = meta.get(
            "approval_status",
            "UNKNOWN",
        )

        version = meta.get(
            "version",
            "unknown",
        )

        role = meta.get(
            "document_role",
            "",
        )

        file_name = meta.get(
            "file_name",
            "",
        )

        score = item.score

        header = (
            f"[SOURCE {i} | "
            f"{document_id} | "
            f"status={status} | "
            f"version={version} | "
            f"role={role}]"
        )

        context_blocks.append(
            f"{header}\n"
            f"{item.node.get_content()}"
        )

        score_text = (
            f"{score:.3f}"
            if score is not None
            else "n/a"
        )

        source_lines.append(
            f"- **{document_id}** — "
            f"{status}, v{version}, "
            f"score {score_text}, "
            f"file `{file_name}`"
        )

    return (
        "\n\n".join(context_blocks),
        "\n".join(source_lines),
    )


print("Context builder ready.")


Context builder ready.


## 13. Final answer function

This is the complete RAG pipeline for one question:

```text
Question
 ↓
Vector retrieval
 ↓
Governance filtering
 ↓
Approved evidence context
 ↓
Load persistent memory
 ↓
System prompt + memory + evidence + question
 ↓
Low-temperature LLM
 ↓
Answer
 ↓
Save new conversation memory
```


In [0]:
def answer_question(
    question: str,
    memory_key: str = "demo",
) -> Tuple[str, str]:

    question = (question or "").strip()

    if not question:
        return "Please enter a question.", ""

    # --------------------------------------------------------
    # A. Semantic retrieval
    # --------------------------------------------------------
    raw_nodes = retriever.retrieve(question)

    # --------------------------------------------------------
    # B. Authority / governance filtering
    # --------------------------------------------------------
    governed_nodes = apply_governance(
        question,
        raw_nodes,
    )

    if not governed_nodes:
        return (
            "I do not have enough approved evidence in the retrieved "
            "context to answer this safely.",
            "No governed sources were selected.",
        )

    # --------------------------------------------------------
    # C. Build evidence context
    # --------------------------------------------------------
    context_text, source_markdown = build_context(
        governed_nodes
    )

    # --------------------------------------------------------
    # D. Persistent conversation memory
    # --------------------------------------------------------
    memory = load_memory(memory_key)

    memory_text = memory_as_text(memory)

    # --------------------------------------------------------
    # E. Final grounded prompt
    # --------------------------------------------------------
    prompt = f"""
SYSTEM INSTRUCTIONS
-------------------
{SYSTEM_PROMPT}

CONVERSATION MEMORY
-------------------
{memory_text}

IMPORTANT:
Conversation memory is context only.
It is NOT evidence and must never override retrieved source documents.

RETRIEVED GOVERNED CONTEXT
--------------------------
{context_text}

USER QUESTION
-------------
{question}

Write the answer now.
""".strip()

    # --------------------------------------------------------
    # F. Generate answer
    # --------------------------------------------------------
    response = llm.complete(prompt)

    answer = getattr(
        response,
        "text",
        str(response),
    ).strip()

    # --------------------------------------------------------
    # G. Persist latest conversation turns
    # --------------------------------------------------------
    updated_memory = memory + [
        {
            "role": "user",
            "content": question,
        },
        {
            "role": "assistant",
            "content": answer,
        },
    ]

    save_memory(
        memory_key,
        updated_memory,
    )

    return answer, source_markdown


print("Complete governed RAG answer function ready.")


Complete governed RAG answer function ready.


## 14. Test the complete RAG system

Start with the most important governance test.


In [0]:
answer, sources = answer_question(
    "When did the vibrations of A-001 degrade? What were the recommendations provided?",
    # "What is the approved inspection procedure for reviewing rising vibration on A-001?",
    memory_key="training_demo",
)

print("ANSWER")
print("=" * 80)
print(answer)

print("\nRETRIEVED GOVERNED SOURCES")
print("=" * 80)
print(sources)


2026-09-22 10:55:35,688 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:55:40,881 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/chat/completions "HTTP/1.1 200 OK"


ANSWER
Answer:
The vibrations of A-001 started to degrade in the August monitoring window, with a sustained increase in vibration compared to its earlier-month baseline. The recommendations provided include applying the approved inspection procedure A001-PROC-INS-001-V2, verifying sensor quality, reviewing the open inspection work order, comparing the latest observation with prior findings, and requesting a qualified reliability review.

Evidence:
- The August condition monitoring report shows a sustained increase in A-001 vibration compared to its earlier-month baseline [A001-CMR-2026-08]
- The vibration trend is the strongest consistent signal in the report period [A001-CMR-2026-08]
- The report recommends applying A001-PROC-INS-001-V2, verifying sensor quality, and reviewing the open inspection work order [A001-CMR-2026-08]
- The inspection work order WO-2026-0817 was created to review the elevated vibration trend [WO-2026-0817]

Interpretation:
The evidence suggests that the vibrat

### A few more good test questions


In [0]:
TEST_QUESTIONS = [
    "What is asset A-001 used for?",
    "What happened to A-001 vibration during August?",
    "What work does WO-2026-0817 currently request?",
    "What was observed in the field on August 19?",
    "Can one high vibration reading diagnose a failure?",
    "What should happen if structured data conflicts with a written report?",
    "Is the V3 draft the current approved inspection procedure?",
    "How did the superseded V1 procedure differ from V2?",
]

for q in TEST_QUESTIONS:
    print("\n" + "=" * 100)
    print("QUESTION:", q)

    answer, sources = answer_question(
        q,
        memory_key="training_demo",
    )

    print("\nANSWER:")
    print(answer)

    print("\nSOURCES:")
    print(sources)



QUESTION: What is asset A-001 used for?


2026-09-22 10:56:48,612 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:56:50,869 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/chat/completions "HTTP/1.1 200 OK"



ANSWER:
Answer:
Asset A-001 is used for closed-loop process cooling support as a cooling water pump.

Evidence:
- A-001 is a cooling water pump serving a fictional industrial utility loop [A001-OPS-001]
- The asset provides closed-loop process cooling support [A001-OPS-001]

Interpretation:
The evidence suggests that A-001 plays a critical role in the industrial utility loop by providing cooling support.

Next human review:
None explicitly indicated, as the question is about the asset's intended use, which is clearly defined in the operating guide.

SOURCES:
- **A001-OPS-001** — APPROVED, v1.4, score 0.505, file `02_A001-OPS-001_Operating_and_Usage_Guide.pdf`
- **A001-OPS-001** — APPROVED, v1.4, score 0.490, file `02_A001-OPS-001_Operating_and_Usage_Guide.pdf`
- **A001-OPS-001** — APPROVED, v1.4, score 0.480, file `02_A001-OPS-001_Operating_and_Usage_Guide.pdf`
- **A001-CMR-2026-08** — APPROVED, v1.0, score 0.453, file `07_A001-CMR-2026-08_Condition_Monitoring_Report.pdf`
- **A001-PRO

2026-09-22 10:56:51,407 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:56:56,099 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/chat/completions "HTTP/1.1 200 OK"



ANSWER:
Answer:
The vibration of A-001 increased during August, with a sustained upward trend compared to its earlier-month baseline.

Evidence:
- The August condition monitoring report shows a sustained increase in A-001 vibration compared to its earlier-month baseline [A001-CMR-2026-08]
- The vibration trend is the strongest consistent signal in the report period [A001-CMR-2026-08]
- Specific vibration readings are provided in the report, including 3.8 mm/s on 2026-08-12, 4.4 mm/s on 2026-08-15, 5.1 mm/s on 2026-08-18, 5.8 mm/s on 2026-08-21, 6.4 mm/s on 2026-08-24, and 6.6 mm/s on 2026-08-25 [A001-CMR-2026-08]

Interpretation:
The evidence suggests that the vibration of A-001 increased during August, with a sustained upward trend. This increase is the strongest consistent signal in the report period.

Next human review:
The evidence indicates that a qualified reliability review is recommended to investigate the cause of the increased vibration and to determine the next course of ac

...


ANSWER:
Answer:
When structured data conflicts with a written report, the conflict should be preserved and stated clearly.

Evidence:
- The troubleshooting guide A001-TSG-001 reminds users to retrieve underlying measurements before acting on a high health-risk score [A001-TSG-001]
- The reliability policy REL-POL-001 states that evidence should be reviewed and any conflicts or missing information should be noted [REL-POL-001]
- The approved inspection procedure A001-PROC-INS-001-V2 requires the reviewer to produce a short evidence briefing containing observed facts, trend summary, and uncertainty or missing evidence [A001-PROC-INS-001-V2]

Interpretation:
The evidence suggests that when structured data conflicts with a written report, the conflict should be preserved and stated clearly, and the underlying measurements should be reviewed to resolve the discrepancy.

Next human review:
The evidence indicates that a qualified human reviewer should be involved to resolve the conflict and 

2026-09-22 10:57:16,692 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:57:20,316 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/chat/completions "HTTP/1.1 200 OK"
2026-09-22 10:57:20,565 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"



ANSWER:
Answer:
No, the V3 draft is not the current approved inspection procedure.

Evidence:
- The current approved inspection procedure is A001-PROC-INS-001-V2 [A001-PROC-INS-001-V2]
- A001-PROC-INS-001-V3D is a draft and has not been approved for use [A001-PROC-INS-001-V3D]
- The draft procedure explicitly states that it is under review and not approved [A001-PROC-INS-001-V3D]

Interpretation:
The evidence suggests that the V3 draft is not the current approved inspection procedure, and A001-PROC-INS-001-V2 remains the authoritative procedure.

Next human review:
None explicitly indicated, as the question is about the status of the inspection procedure, which is clearly defined in the retrieved documents. However, it is essential to note that the V3 draft is under review and may become the approved procedure in the future, but currently, V2 is the approved version to be followed [A001-PROC-INS-001-V2, A001-PROC-INS-001-V3D].

SOURCES:
- **A001-PROC-INS-001-V2** — APPROVED, v2.0, sco

2026-09-22 10:57:24,768 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/chat/completions "HTTP/1.1 200 OK"



ANSWER:
Answer:
The superseded V1 procedure differed from V2 in that it did not explicitly require trend comparison, document-version verification, or evidence-gap recording.

Evidence:
- Version 1.0 did not explicitly require trend comparison, document-version verification, or evidence-gap recording [A001-PROC-INS-001-V1]
- These controls were added in version 2.0 after internal lessons learned [A001-PROC-INS-001-V1]
- The approved inspection procedure A001-PROC-INS-001-V2 requires the reviewer to produce a short evidence briefing containing observed facts, trend summary, relevant work history, current procedure citation, uncertainty or missing evidence, and the requested next human review [A001-PROC-INS-001-V2]

Interpretation:
The evidence suggests that the superseded V1 procedure had limitations that were addressed in V2, including the addition of trend comparison, document-version verification, and evidence-gap recording requirements.

Next human review:
None explicitly indicated

## 15. Test persistent memory

Ask a question, then a follow-up that depends on the previous turn.

The previous turn helps provide conversational context, but the answer must still be grounded in retrieved evidence.


In [0]:
clear_memory("memory_demo")

answer_1, _ = answer_question(
    "What is the current approved inspection procedure for A-001?",
    memory_key="memory_demo",
)

print("TURN 1")
print(answer_1)

print("\n" + "=" * 100 + "\n")

answer_2, _ = answer_question(
    "Why should I not use the previous version as current guidance?",
    memory_key="memory_demo",
)

print("TURN 2")
print(answer_2)

print("\n" + "=" * 100 + "\n")

print("PERSISTED MEMORY")
print(
    json.dumps(
        load_memory("memory_demo"),
        indent=2,
    )
)


2026-09-22 10:57:26,238 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:57:29,004 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/chat/completions "HTTP/1.1 200 OK"


TURN 1
Answer:
The current approved inspection procedure for A-001 is A001-PROC-INS-001-V2.

Evidence:
- A001-PROC-INS-001-V2 is the current approved inspection procedure [A001-PROC-INS-001-V2]
- The document status is APPROVED and the version is 2.0 [A001-PROC-INS-001-V2]
- The procedure defines the training workflow for reviewing an abnormal condition on A-001 [A001-PROC-INS-001-V2]

Interpretation:
The evidence supports that A001-PROC-INS-001-V2 is the current approved inspection procedure for A-001, which outlines the steps for reviewing an abnormal condition on the asset.

Next human review:
None required for this specific question, as the evidence clearly states the current approved inspection procedure.




2026-09-22 10:57:29,516 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 10:57:34,129 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/chat/completions "HTTP/1.1 200 OK"


TURN 2
Answer:
You should not use the previous version (A001-PROC-INS-001-V1) as current guidance because it is superseded by A001-PROC-INS-001-V2.

Evidence:
- A001-PROC-INS-001-V1 is superseded [A001-PROC-INS-001-V2]
- The current approved inspection procedure for A-001 is A001-PROC-INS-001-V2 [A001-PROC-INS-001-V2]
- Source authority rules state that A001-PROC-INS-001-V1 is superseded and should not be used as current guidance [SYSTEM INSTRUCTIONS]

Interpretation:
The evidence supports that A001-PROC-INS-001-V1 is no longer the current approved inspection procedure and has been superseded by A001-PROC-INS-001-V2, which should be used for current guidance.

Next human review:
None required for this specific question, as the evidence clearly states the supersession of A001-PROC-INS-001-V1.


PERSISTED MEMORY
[
  {
    "role": "user",
    "content": "What is the current approved inspection procedure for A-001?"
  },
  {
    "role": "assistant",
    "content": "Answer:\nThe current app

## 16. Simple Gradio application

The UI contains:

- persistent-memory key
- chat window
- question box
- Ask button
- retrieved governed sources
- load memory
- clear memory
- clear screen

This keeps the interface simple enough for a live training demo.


In [0]:
def submit_question(
    question,
    history,
    memory_key,
):
    history = history or []

    answer, sources = answer_question(
        question=question,
        memory_key=memory_key,
    )

    history = history + [
        {
            "role": "user",
            "content": question,
        },
        {
            "role": "assistant",
            "content": answer,
        },
    ]

    return "", history, sources


def load_memory_into_ui(memory_key):
    history = load_memory(memory_key)

    return (
        history,
        "Persistent memory loaded.",
    )


def clear_memory_ui(memory_key):
    clear_memory(memory_key)

    return (
        [],
        "Persistent memory cleared.",
        "",
    )


with gr.Blocks(
    title="A-001 Governed RAG Assistant"
) as demo:

    gr.Markdown(
        """
# A-001 Governed RAG Assistant

**PDFs → chunks → embeddings → persistent index → governed retrieval → LLM → cited answer**

For current inspection guidance, the application is designed to use
**A001-PROC-INS-001-V2** and reject Draft/Superseded procedures unless
the user explicitly asks about those versions.

> Synthetic training application only.
"""
    )

    with gr.Row():

        memory_key = gr.Textbox(
            value="demo",
            label="Memory key",
            info=(
                "Use another key to create a separate "
                "persistent conversation."
            ),
        )

        load_memory_btn = gr.Button(
            "Load memory"
        )

        clear_memory_btn = gr.Button(
            "Clear memory"
        )

    chatbot = gr.Chatbot(
        label="Question & Answer",
        height=480,
    )

    question = gr.Textbox(
        label="Ask a question",
        placeholder=(
            "Example: What is the approved inspection "
            "procedure for reviewing rising vibration on A-001?"
        ),
        lines=2,
    )

    with gr.Row():

        ask_btn = gr.Button(
            "Ask",
            variant="primary",
        )

        clear_chat_btn = gr.Button(
            "Clear screen"
        )

    gr.Markdown(
        "### Retrieved governed sources"
    )

    source_box = gr.Markdown()

    status_box = gr.Markdown()

    ask_btn.click(
        fn=submit_question,
        inputs=[
            question,
            chatbot,
            memory_key,
        ],
        outputs=[
            question,
            chatbot,
            source_box,
        ],
    )

    question.submit(
        fn=submit_question,
        inputs=[
            question,
            chatbot,
            memory_key,
        ],
        outputs=[
            question,
            chatbot,
            source_box,
        ],
    )

    load_memory_btn.click(
        fn=load_memory_into_ui,
        inputs=[memory_key],
        outputs=[
            chatbot,
            status_box,
        ],
    )

    clear_memory_btn.click(
        fn=clear_memory_ui,
        inputs=[memory_key],
        outputs=[
            chatbot,
            status_box,
            source_box,
        ],
    )

    clear_chat_btn.click(
        fn=lambda: ([], ""),
        outputs=[
            chatbot,
            source_box,
        ],
    )

print("Gradio UI created.")


2026-09-22 11:18:01,426 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"


Gradio UI created.


## 17. Launch the Gradio app

### Option A — easiest for a notebook demo

Try:

```python
demo.launch(share=True)
```

A public Gradio share URL requires outbound network access.

### Option B — local/server launch

```python
demo.launch(
    server_name="0.0.0.0",
    server_port=7860
)
```

Which launch method works best depends on how your Databricks environment exposes notebook-hosted web servers.

For training, I recommend first running the entire RAG pipeline in notebook cells.  
Then use the Gradio UI as the final “put everything together” demo.


In [0]:
# For a notebook demo, uncomment ONE launch option.

# OPTION 1:
demo.launch(share=True)

# OPTION 2:
# demo.launch(
#     server_name="0.0.0.0",
#     server_port=7860,
# )


2026-09-22 11:07:37,534 - INFO - HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7860


2026-09-22 11:07:37,814 - INFO - HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
2026-09-22 11:07:38,056 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
2026-09-22 11:07:38,184 - INFO - HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_arm64 "HTTP/1.1 200 OK"


* Running on public URL: https://f256e90f4d3cc0751f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-09-22 11:07:38,728 - INFO - HTTP Request: HEAD https://f256e90f4d3cc0751f.gradio.live "HTTP/1.1 200 OK"


2026-09-22 11:07:38,856 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


In [0]:
with gr.Blocks(title="A-001 RAG") as app:
    memory_key = gr.Textbox(value="demo", label="Memory key")
    chatbot = gr.Chatbot(label="Conversation", height=400)
    question = gr.Textbox(label="Ask", placeholder="Type your question…", lines=2)
    ask_btn = gr.Button("Ask", variant="primary")

    def ask(q, history, mk):
        history = history or []
        ans, _ = answer_question(question=q, memory_key=mk)
        return "", history + [{"role": "user", "content": q}, {"role": "assistant", "content": ans}]

    ask_btn.click(ask, [question, chatbot, memory_key], [question, chatbot])
    question.submit(ask, [question, chatbot, memory_key], [question, chatbot])

app.launch(share=True)

2026-09-22 11:20:25,086 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-09-22 11:20:25,266 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-09-22 11:20:25,267 - INFO - HTTP Request: GET http://127.0.0.1:7863/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-09-22 11:20:25,279 - INFO - HTTP Request: HEAD http://127.0.0.1:7863/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7863


2026-09-22 11:20:25,518 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


* Running on public URL: https://64f8258008e3184422.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-09-22 11:20:26,136 - INFO - HTTP Request: HEAD https://64f8258008e3184422.gradio.live "HTTP/1.1 200 OK"


2026-09-22 11:20:26,192 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


Gradio version: 6.28.0
Chatbot created without type param: OK
Chatbot signature params: ['self', 'value', 'label', 'every', 'inputs', 'show_label', 'container', 'scale', 'min_width', 'visible']


# Teaching story — what to explain to participants

## Part 1 — Simple RAG

```text
Documents
 ↓
Chunks
 ↓
Embeddings
 ↓
Vector index
 ↓
Retrieve
 ↓
LLM answer
```

This solves the basic problem:

> “How can an LLM answer from my private documents?”

---

## Part 2 — Persistence

Without persistence:

```text
Restart notebook
 ↓
reload PDFs
 ↓
recreate chunks
 ↓
recreate embeddings
```

With persistence:

```text
Restart notebook
 ↓
load saved LlamaIndex store
```

---

## Part 3 — Memory

Simple RAG treats every question independently.

Memory lets the assistant understand follow-up questions.

But:

> **Memory is context, not evidence.**

---

## Part 4 — Enterprise governance

Similarity alone is not enough.

A superseded procedure may be extremely similar to the current procedure.

A draft may even have a higher version number.

So the system needs:

```text
Retrieval
   +
Approval status
   +
Version
   +
Document role
   +
Authority rules
```

---

## Part 5 — System prompt

The system prompt tells the model how to behave **after** governed evidence is selected.

It controls:

- grounding
- citation style
- uncertainty
- separation of facts and interpretation
- human review
- safety boundaries

But we do not rely on the prompt alone.

The stronger design is:

```text
Governance in CODE
        +
Governance in PROMPT
```

---

## Final architecture

```text
                     ┌──────────────────────┐
                     │      A-001 PDFs      │
                     └──────────┬───────────┘
                                ↓
                     ┌──────────────────────┐
                     │      Chunking        │
                     └──────────┬───────────┘
                                ↓
                     ┌──────────────────────┐
                     │     Embeddings       │
                     └──────────┬───────────┘
                                ↓
                     ┌──────────────────────┐
                     │ Persistent Vector DB │
                     └──────────┬───────────┘
                                ↓
                     ┌──────────────────────┐
                     │ Semantic Retrieval   │
                     └──────────┬───────────┘
                                ↓
                  ┌────────────────────────────┐
                  │ Governance / Authority     │
                  │ Approved / Draft / Old     │
                  └─────────────┬──────────────┘
                                ↓
              ┌─────────────────┴─────────────────┐
              │                                   │
              ↓                                   ↓
   ┌─────────────────────┐             ┌─────────────────────┐
   │ Retrieved Evidence  │             │ Persistent Memory   │
   └──────────┬──────────┘             └──────────┬──────────┘
              │                                   │
              └─────────────────┬─────────────────┘
                                ↓
                     ┌──────────────────────┐
                     │    System Prompt     │
                     └──────────┬───────────┘
                                ↓
                     ┌──────────────────────┐
                     │   Databricks LLM     │
                     │  Temperature = 0.1   │
                     └──────────┬───────────┘
                                ↓
                     ┌──────────────────────┐
                     │ Cited / Grounded     │
                     │ Answer               │
                     └──────────┬───────────┘
                                ↓
                     ┌──────────────────────┐
                     │      Gradio UI       │
                     └──────────────────────┘
```

## One-line takeaway

> **Enterprise RAG = retrieval + authority + memory + evidence + governance + human review.**
